# CEM 有理数精确真值：从输入到认证 / Exact rational CEM truth from inputs to certification

## 目标 / Goal

| 中文 | English |
|---|---|
| 不读取一个神秘的“小数真值”，而是从 X01 的有理坐标、整数拓扑、有理单元电导率、有理接触阻抗和整数电流开始，在 $\mathbb{Q}$ 上重新组装 CEM，并逐步得到经典 CEM 与 Robin CEM 完全相同的分数电极电压。 | Instead of loading a mysterious decimal “truth”, start from X01 rational coordinates, integer topology, rational cell conductivities, rational contact impedance, and integer currents; reassemble the CEM over $\mathbb{Q}$ and obtain exactly identical fractional electrode voltages from Classic and Robin CEM. |
| 最后把真实 `float64` PyEIDORS 电压与该分数矩阵比较，展示报告中的真值误差和缩放后向残差如何得到。 | Finally compare the actual `float64` PyEIDORS voltage with that fraction matrix and reproduce the truth error and scaled backward residual used by the report. |


## 真值的适用范围 / Scope of the truth

| 中文 | English |
|---|---|
| 这里的“数学精确真值”是**固定有理 P1 有限维 CEM 线性系统的唯一精确解**，不是连续 PDE 的解析真值，也不是光滑真实圆域的解析解。 | Here “mathematical exact truth” means the **unique exact solution of one fixed rational finite-dimensional P1 CEM system**; it is not the analytic truth of the continuum PDE and not an analytic solution on a smooth physical disk. |
| 因为三个框架使用同一节点、单元、电极边、$\sigma,z,I$ 和 P1 离散，所以该真值可以隔离并比较组装/线性代数的浮点误差。若要研究连续物理误差，还必须另外做真实圆域网格加密与独立高阶参考实验。 | Because all three frameworks use the same nodes, cells, electrode edges, $\sigma,z,I$, and P1 discretization, this truth isolates floating-point assembly/linear-algebra error. Continuous-physics error requires a separate true-disk refinement study with an independent high-order reference. |


## 设置 / Setup

| 中文 | English |
|---|---|
| 使用与 PyEIDORS Notebook 相同的真实数 `float64` Nix 内核；SymPy 只负责 $\mathbb{Q}$ 精确代数，DOLFINx 只用于生成待比较的 `float64` 结果。 | Use the same real `float64` Nix kernel as the PyEIDORS notebook. SymPy performs exact algebra over $\mathbb{Q}$; DOLFINx is used only to generate the `float64` candidate being compared. |

```bash
nix develop .#default --command jupyter lab \
  examples/cem_exact_extension_walkthrough/exact_rational_truth_walkthrough.ipynb
```


In [1]:
from fractions import Fraction
import hashlib
import json
from pathlib import Path
import sys
from types import SimpleNamespace

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "cem_exact_extension_walkthrough":
    NOTEBOOK_DIR = Path("examples/cem_exact_extension_walkthrough").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
for path in (REPO_ROOT, REPO_ROOT / "src", NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib import font_manager  # noqa: E402
from sympy import Matrix, QQ, SparseMatrix  # noqa: E402

from experiment_common import (  # noqa: E402
    build_classic_state,
    build_robin_state,
    exact_reference_metrics,
    load_assembled_blocks,
    load_forward_fixture,
    load_portable_exact_reference,
    plot_forward_fixture,
    plot_forward_solution,
    solve_classic,
    solve_robin,
)
from pyeidors_debug import ensure_pyeidors_case  # noqa: E402
from scripts.benchmarks.cem_exact_extension_suite import (  # noqa: E402
    EXTENSION_CASES,
    _exact_currents,
    _zero_sum_basis,
    assemble_exact_extension_cem,
    extension_case_cell_conductivities,
    extension_case_mesh,
    extension_current_patterns,
)

for font_path in (
    Path("/mnt/c/Windows/Fonts/times.ttf"),
    Path("/mnt/c/Windows/Fonts/timesbd.ttf"),
    Path("/mnt/c/Windows/Fonts/msyh.ttc"),
):
    if font_path.exists():
        font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["Times New Roman", "Microsoft YaHei"]

In [2]:
# 教学案例和路径 / Teaching case and paths.
CASE_ID = "X01"
REGENERATE_FLOAT = False
SUITE_OUTPUT = REPO_ROOT / "output" / "cem_exact_extension"
PORTABLE_REFERENCE_PATH = NOTEBOOK_DIR / "fixtures" / CASE_ID / "exact_reference.json"
PORTABLE_FIXTURE_DIR = NOTEBOOK_DIR / "fixtures" / CASE_ID / "common_mesh"
FIGURE_DIR = NOTEBOOK_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 符号与变量字典 / Symbol and variable dictionary

| 变量或符号 | 中文含义 | English meaning | 类型或维度 |
|---|---|---|---|
| `case` | X01 的预注册物理/离散设置。 | Preregistered physical/discrete settings for X01. | `ExtensionCase` |
| `exact_nodes` | 不经过浮点舍入的节点坐标，每个坐标都是 `Fraction`。 | Node coordinates before floating-point rounding; every coordinate is a `Fraction`. | $N\times2$ |
| `cells` | P1 三角形拓扑，只有整数节点索引。 | P1 triangle topology containing only integer node indices. | $K\times3$ |
| `tagged_edges` | 边界边与电极标签。 | Boundary edges and electrode labels. | $E_b\times3$ |
| `cell_sigma`, $\sigma_k$ | 第 $k$ 个三角形的精确有理电导率。 | Exact rational conductivity of triangle $k$. | $K$ 个 `Fraction` |
| `case.contact_impedance`, $z$ | 精确有理接触阻抗。 | Exact rational contact impedance. | `Fraction` |
| `I_QQ`, $I$ | 精确整数注入电流矩阵，每列总和为零。 | Exact integer drive-current matrix; every column sums to zero. | $L$×$P$ |
| `A_R_QQ`, $A_R$ | 精确体刚度加 Robin 边界质量矩阵。 | Exact body stiffness plus Robin boundary mass matrix. | $N$×$N$ over $\mathbb{Q}$ |
| `C_QQ`, $C$ | 精确体节点/电极耦合矩阵。 | Exact body-node/electrode coupling matrix. | $N$×$L$ over $\mathbb{Q}$ |
| `D_QQ`, $D$ | 精确电极边界积分块。 | Exact electrode boundary-integral block. | $L$×$L$ over $\mathbb{Q}$ |
| `B_QQ`, $B$ | 有理零和基，列为 $e_j-e_L$；不用含平方根的正交基，保证所有元素仍在 $\mathbb{Q}$。 | Rational zero-sum basis with columns $e_j-e_L$; it avoids a square-root orthonormal basis so every entry remains in $\mathbb{Q}$. | $L$×$(L-1)$ |
| `classic_matrix_QQ`, $\mathcal{K}$ | 带零均值规范的传统 CEM 增广矩阵。 | Gauge-augmented Classic CEM matrix. | $(N+L+1)$×$(N+L+1)$ |
| `classic_basis_solution_QQ` | 对 $L-1$ 个基电流一次精确多右端求解得到的解基。 | Solution basis from one exact multi-RHS solve for $L-1$ basis currents. | $(N+L+1)$×$(L-1)$ |
| `response_QQ` | $A_R^{-1}C$ 的精确解；这里只是记号，代码调用求解器而不构造浮点逆矩阵。 | Exact solution of $A_R X=C$; the inverse is notation only and no floating inverse is formed. | $N$×$L$ |
| `reduced_map_QQ`, $T_r$ | $B^\mathsf{T}(D-C^\mathsf{T}A_R^{-1}C)B$。 | $B^\mathsf{T}(D-C^\mathsf{T}A_R^{-1}C)B$. | $(L-1)$×$(L-1)$ |
| `U_classic_QQ`, `U_robin_QQ` | 两种精确路径得到的电极电压；必须逐分数完全相同。 | Electrode voltages from the two exact routes; they must be identical fraction by fraction. | $L$×$P$ |
| `truth_sha256` | 对规范分数字符串矩阵计算的哈希，用来证明保存/读取没有改变真值。 | Hash of the canonical fraction-string matrix, proving serialization did not change the truth. | SHA-256 |


## 步骤 / Steps

### 1. 构造并显示完全相同的正问题输入 / Build and display the identical forward input

| 中文 | English |
|---|---|
| X01 有 $N=33$ 个节点、$K=32$ 个三角形、$L=16$ 个电极和 $P=16$ 个相邻电流模式。所有单元的背景电导率为 $\sigma=1/8$，接触阻抗为 $z=1$；没有内部异常物。 | X01 has $N=33$ nodes, $K=32$ triangles, $L=16$ electrodes, and $P=16$ adjacent drive patterns. Every cell has background conductivity $\sigma=1/8$ and contact impedance $z=1$; there is no interior anomaly. |


In [3]:
case = next(item for item in EXTENSION_CASES if item.case_id == CASE_ID)
exact_nodes, cells, tagged_edges, electrode_nodes, electrode_counts = (
    extension_case_mesh(case)
)
cell_sigma = extension_case_cell_conductivities(case, exact_nodes, cells)
current_patterns_float = extension_current_patterns(
    case.n_electrodes,
    case.drive_skip,
)
I_QQ = _exact_currents(current_patterns_float)

forward_fixture = load_forward_fixture(
    PORTABLE_FIXTURE_DIR / "cem_exact_extension_p1.mat",
    PORTABLE_FIXTURE_DIR / "cem_exact_extension_p1.json",
)
assert np.array_equal(
    forward_fixture.nodes,
    np.asarray([[float(x), float(y)] for x, y in exact_nodes]),
)
assert np.array_equal(forward_fixture.cells, cells)
assert np.array_equal(forward_fixture.tagged_edges, tagged_edges)
assert np.array_equal(
    forward_fixture.cell_conductivity,
    np.asarray(cell_sigma, dtype=np.float64),
)

{
    "case": case.case_id,
    "N_nodes": len(exact_nodes),
    "K_cells": cells.shape[0],
    "L_electrodes": case.n_electrodes,
    "P_current_patterns": I_QQ.cols,
    "conductivity_pattern": case.conductivity_pattern,
    "unique_exact_sigma": sorted(set(cell_sigma)),
    "exact_contact_impedance": case.contact_impedance,
    "mesh_fingerprint": forward_fixture.mesh_fingerprint,
}

{'case': 'X01',
 'N_nodes': 33,
 'K_cells': 32,
 'L_electrodes': 16,
 'P_current_patterns': 16,
 'conductivity_pattern': 'uniform',
 'unique_exact_sigma': [Fraction(1, 8)],
 'exact_contact_impedance': Fraction(1, 1),
 'mesh_fingerprint': '7be7165ad3bdd3661ae06bea768622741ece609acde5720f3dd0c0cbde85c5bc'}

In [4]:
fairness_figure, fairness_axes = plot_forward_fixture(
    forward_fixture,
    current_column=0,
)
fairness_figure.savefig(
    FIGURE_DIR / f"{CASE_ID}_exact_forward_setup.png",
    dpi=180,
    bbox_inches="tight",
)
fairness_figure

<Figure size 1320x580 with 3 Axes>

### 2. 证明输入确实属于有理数域 / Prove that the inputs lie in the rational field

| 中文 | English |
|---|---|
| `Fraction(p,q)` 保存整数分子和非零整数分母，不执行二进制浮点舍入。当前、几何和物性输入都先以这种形式构造；三角形拓扑和电极标签本来就是整数。 | `Fraction(p,q)` stores an integer numerator and a nonzero integer denominator without binary floating-point rounding. Geometry and material inputs are constructed in this form; triangle topology and electrode labels are already integers. |


In [5]:
all_coordinates_are_fraction = all(
    isinstance(value, Fraction) for point in exact_nodes for value in point
)
all_conductivities_are_fraction = all(
    isinstance(value, Fraction) for value in cell_sigma
)
contact_impedance_is_fraction = isinstance(case.contact_impedance, Fraction)
all_currents_are_rational = all(value.is_Rational for value in I_QQ)
all_current_columns_are_exactly_zero_sum = all(
    sum(I_QQ[row, column] for row in range(I_QQ.rows)) == 0
    for column in range(I_QQ.cols)
)
input_certification = {
    "all_coordinates_are_fraction": all_coordinates_are_fraction,
    "all_conductivities_are_fraction": all_conductivities_are_fraction,
    "contact_impedance_is_fraction": contact_impedance_is_fraction,
    "all_currents_are_rational": all_currents_are_rational,
    "all_current_columns_are_exactly_zero_sum": (
        all_current_columns_are_exactly_zero_sum
    ),
    "first_five_exact_nodes": exact_nodes[:5],
    "first_drive": list(I_QQ[:, 0]),
}
input_certification

{'all_coordinates_are_fraction': True,
 'all_conductivities_are_fraction': True,
 'contact_impedance_is_fraction': True,
 'all_currents_are_rational': True,
 'all_current_columns_are_exactly_zero_sum': True,
 'first_five_exact_nodes': ((Fraction(0, 1), Fraction(0, 1)),
  (Fraction(11113, 16384), Fraction(2861, 65536)),
  (Fraction(10595, 16384), Fraction(13715, 65536)),
  (Fraction(9895, 16384), Fraction(20435, 65536)),
  (Fraction(8333, 16384), Fraction(29549, 65536))),
 'first_drive': [1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

### 3. 用精确 P1 积分组装 $A_R,C,D$ / Assemble $A_R,C,D$ using exact P1 integrals

对三角形 $K$，P1 梯度为常数，因此体刚度条目可以只用有理数加减乘除：

$$
(A_\Omega^K)_{ij}
=\sigma_K\int_K\nabla\phi_i\cdot\nabla\phi_j\,dx
=\sigma_K\frac{b_i b_j+c_i c_j}{4|K|}.
$$

对一条长度为 $|e|$、接触阻抗为 $z_\ell$ 的电极边，精确边界条目为：

$$
\frac{|e|}{z_\ell}
\begin{bmatrix}
1/3 & 1/6\\
1/6 & 1/3
\end{bmatrix},
\qquad
C_{e,\ell}=-\frac{|e|}{2z_\ell}
\begin{bmatrix}1\\1\end{bmatrix},
\qquad
D_{\ell\ell}\mathrel{+}=\frac{|e|}{z_\ell}.
$$

| 中文 | English |
|---|---|
| 该专用有理网格保证所用电极边长也是可精确开平方的有理数；若不是，组装函数会立即拒绝案例。 | This purpose-built rational mesh guarantees rational electrode-edge lengths with exact square roots; assembly immediately rejects a case that violates this property. |


In [6]:
A_R_QQ, C_QQ, D_QQ = assemble_exact_extension_cem(
    exact_nodes,
    cells,
    tagged_edges,
    cell_conductivities=cell_sigma,
    contact_impedance=case.contact_impedance,
    n_electrodes=case.n_electrodes,
)
block_domains = {
    "A_R_domain": str(A_R_QQ.to_DM().convert_to(QQ).domain),
    "C_domain": str(C_QQ.to_DM().convert_to(QQ).domain),
    "D_domain": str(D_QQ.to_DM().convert_to(QQ).domain),
    "A_R_shape": A_R_QQ.shape,
    "C_shape": C_QQ.shape,
    "D_shape": D_QQ.shape,
    "A_R_sample_fraction": A_R_QQ[0, 0],
    "C_sample_fraction": C_QQ[0, 0],
    "D_sample_fraction": D_QQ[0, 0],
}
block_domains

{'A_R_domain': 'QQ',
 'C_domain': 'QQ',
 'D_domain': 'QQ',
 'A_R_shape': (33, 33),
 'C_shape': (33, 16),
 'D_shape': (16, 16),
 'A_R_sample_fraction': 115795767021/293709566584,
 'C_sample_fraction': 0,
 'D_sample_fraction': 5525/32768}

### 4. 传统 CEM：在 $\mathbb{Q}$ 上精确 LU / Classic CEM: exact LU over $\mathbb{Q}$

$$
\mathcal{K}
=
\begin{bmatrix}
A_R & C & 0\\
C^\mathsf{T} & D & \mathbf{1}\\
0 & \mathbf{1}^\mathsf{T} & 0
\end{bmatrix},
\qquad
\mathcal{K}
\begin{bmatrix}u\\U\\\lambda\end{bmatrix}
=
\begin{bmatrix}0\\I\\0\end{bmatrix}.
$$

| 中文 | English |
|---|---|
| 为避免逐个电流模式重复分解，先对有理零和基 $B=[e_1-e_L,\ldots,e_{L-1}-e_L]$ 的 $L-1$ 个右端一起求解。因为每列 $I$ 的和为零，所以 $I=B\,I_{1:L-1,:}$。 | To avoid repeated factorizations, solve the $L-1$ right-hand sides of the rational zero-sum basis $B=[e_1-e_L,\ldots,e_{L-1}-e_L]$ together. Since every column of $I$ sums to zero, $I=B\,I_{1:L-1,:}$. |
| `DomainMatrix.convert_to(QQ).lu_solve(...)` 只执行整数/分数运算；没有容差、没有舍入、没有“足够接近零”。 | `DomainMatrix.convert_to(QQ).lu_solve(...)` performs only integer/fraction operations: no tolerance, no rounding, and no “close enough to zero”. |


In [7]:
N = len(exact_nodes)
L = case.n_electrodes
P = I_QQ.cols
B_QQ = _zero_sum_basis(L)
assert B_QQ.T * Matrix.ones(L, 1) == Matrix.zeros(L - 1, 1)

classic_size = N + L + 1
classic_matrix_QQ = SparseMatrix.zeros(classic_size, classic_size)
classic_matrix_QQ[:N, :N] = A_R_QQ
classic_matrix_QQ[:N, N : N + L] = C_QQ
classic_matrix_QQ[N : N + L, :N] = C_QQ.T
classic_matrix_QQ[N : N + L, N : N + L] = D_QQ
for electrode in range(L):
    classic_matrix_QQ[N + electrode, classic_size - 1] = 1
    classic_matrix_QQ[classic_size - 1, N + electrode] = 1

classic_basis_rhs_QQ = SparseMatrix.zeros(classic_size, L - 1)
classic_basis_rhs_QQ[N : N + L, :] = B_QQ
classic_basis_solution_QQ = (
    classic_matrix_QQ.to_DM()
    .convert_to(QQ)
    .lu_solve(classic_basis_rhs_QQ.to_DM().convert_to(QQ))
    .to_Matrix()
)
# A square singular matrix makes exact DomainMatrix.lu_solve raise instead
# of returning a solution. Reaching this line therefore certifies invertibility.
classic_exact_lu_succeeded = True
classic_matrix_has_full_exact_rank = (
    classic_matrix_QQ.rows == classic_matrix_QQ.cols and classic_exact_lu_succeeded
)
classic_basis_residual_QQ = (
    classic_matrix_QQ * classic_basis_solution_QQ - classic_basis_rhs_QQ
)
classic_residual_is_exact_zero = all(value == 0 for value in classic_basis_residual_QQ)

current_coordinates_QQ = I_QQ[: L - 1, :]
assert B_QQ * current_coordinates_QQ == I_QQ
classic_solution_QQ = classic_basis_solution_QQ * current_coordinates_QQ
classic_rhs_QQ = SparseMatrix.zeros(classic_size, P)
classic_rhs_QQ[N : N + L, :] = I_QQ
assert classic_matrix_QQ * classic_solution_QQ == classic_rhs_QQ
U_classic_QQ = classic_solution_QQ[N : N + L, :]

{
    "classic_matrix_domain": str(classic_matrix_QQ.to_DM().convert_to(QQ).domain),
    "classic_matrix_shape": classic_matrix_QQ.shape,
    "basis_rhs_shape": classic_basis_rhs_QQ.shape,
    "classic_residual_is_exact_zero": classic_residual_is_exact_zero,
    "sample_exact_voltage": U_classic_QQ[0, 0],
}

{'classic_matrix_domain': 'QQ',
 'classic_matrix_shape': (50, 50),
 'basis_rhs_shape': (50, 15),
 'classic_residual_is_exact_zero': True,
 'sample_exact_voltage': 312885894500615529825076242659692029638928471469201423682228591917591410971743666257568061576998280114317372191813779120571453914833551828816/40905254528926752731819321131009679145161324295167080484911081929166684238167393892820909737580954252600062584379209839241150477200981027175}

### 5. Robin CEM：精确消去体未知量 / Robin CEM: eliminate body unknowns exactly

$$
T=D-C^\mathsf{T}A_R^{-1}C,
\qquad
T_r=B^\mathsf{T}TB.
$$

$$
T_r y=B^\mathsf{T}I,
\qquad
U=By,
\qquad
u=-A_R^{-1}CU.
$$

| 中文 | English |
|---|---|
| 代码仍然不显式构造逆矩阵；`lu_solve(C)` 表示精确求解 $A_RX=C$。Robin 路径的乘法/消元顺序与传统增广路径不同，因此它是独立的代数交叉认证。 | The code still does not form an inverse; `lu_solve(C)` means solving $A_RX=C$ exactly. Robin uses a different elimination and multiplication order from the augmented route, providing an independent algebraic cross-certification. |


In [8]:
response_QQ = (
    A_R_QQ.to_DM().convert_to(QQ).lu_solve(C_QQ.to_DM().convert_to(QQ)).to_Matrix()
)
transconductance_QQ = D_QQ - C_QQ.T * response_QQ
reduced_map_QQ = B_QQ.T * transconductance_QQ * B_QQ
reduced_rhs_QQ = B_QQ.T * I_QQ
robin_coefficients_QQ = (
    reduced_map_QQ.to_DM()
    .convert_to(QQ)
    .lu_solve(reduced_rhs_QQ.to_DM().convert_to(QQ))
    .to_Matrix()
)
# The reduced map is square; successful exact LU certifies that it has rank L-1.
robin_exact_lu_succeeded = True
reduced_map_has_full_exact_rank = (
    reduced_map_QQ.rows == reduced_map_QQ.cols and robin_exact_lu_succeeded
)
robin_residual_QQ = reduced_map_QQ * robin_coefficients_QQ - reduced_rhs_QQ
robin_residual_is_exact_zero = all(value == 0 for value in robin_residual_QQ)
U_robin_QQ = B_QQ * robin_coefficients_QQ
classic_robin_exactly_identical = U_classic_QQ == U_robin_QQ
exact_voltage_gauge_zero = all(
    sum(U_classic_QQ[row, column] for row in range(L)) == 0 for column in range(P)
)

{
    "reduced_map_domain": str(reduced_map_QQ.to_DM().convert_to(QQ).domain),
    "reduced_map_shape": reduced_map_QQ.shape,
    "robin_residual_is_exact_zero": robin_residual_is_exact_zero,
    "classic_robin_exactly_identical": classic_robin_exactly_identical,
    "exact_voltage_gauge_zero": exact_voltage_gauge_zero,
}

{'reduced_map_domain': 'QQ',
 'reduced_map_shape': (15, 15),
 'robin_residual_is_exact_zero': True,
 'classic_robin_exactly_identical': True,
 'exact_voltage_gauge_zero': True}

### 6. 精确正问题结果可视化 / Visualize the exact forward result

| 中文 | English |
|---|---|
| 把已经在 $\mathbb{Q}$ 上求出的分数解仅为绘图转换成 `float64`。上排显示精确 Classic/Robin 体电势和它们的差值；下排显示精确电极电压。由于两条有理路径严格同解，差值图应为零。 | Convert the already-solved rational fractions to `float64` only for plotting. The top row shows the exact Classic/Robin body fields and their difference; the bottom row shows exact electrode voltages. Since both rational routes are strictly identical, the difference plots must be zero. |
| 这次转换不参与真值生成、哈希或误差计算；精确认证仍使用原始 SymPy `QQ` 分数对象。 | This conversion is not used to generate the truth, hash, or error metrics; certification still uses the original SymPy `QQ` fractions. |


In [9]:
classic_body_QQ = classic_solution_QQ[:N, :]
robin_body_QQ = -(response_QQ * U_robin_QQ)
assert classic_body_QQ == robin_body_QQ

exact_plot_solutions = {
    "classic": SimpleNamespace(
        body_potential=np.asarray(classic_body_QQ, dtype=np.float64),
        electrode_voltage=np.asarray(U_classic_QQ, dtype=np.float64),
    ),
    "robin_transconductance": SimpleNamespace(
        body_potential=np.asarray(robin_body_QQ, dtype=np.float64),
        electrode_voltage=np.asarray(U_robin_QQ, dtype=np.float64),
    ),
}
exact_result_figure, exact_result_axes = plot_forward_solution(
    forward_fixture,
    exact_plot_solutions,
    current_column=0,
)
exact_result_figure.savefig(
    FIGURE_DIR / f"{CASE_ID}_exact_classic_robin_results.png",
    dpi=180,
    bbox_inches="tight",
)
exact_result_figure

<Figure size 1580x940 with 8 Axes>

### 7. 证明解唯一并认证保存的分数真值 / Prove uniqueness and certify the stored fraction truth

唯一性与精确性的证据是：

$$
\operatorname{rank}_{\mathbb{Q}}(\mathcal{K})=N+L+1,
\qquad
\operatorname{rank}_{\mathbb{Q}}(T_r)=L-1,
$$

$$
\mathcal{K}X-B_{\mathrm{rhs}}=0,\qquad
T_rY-B^\mathsf{T}I=0,\qquad
U_{\mathrm{Classic}}-U_{\mathrm{Robin}}=0.
$$

| 中文 | English |
|---|---|
| 上式中的零是 SymPy 有理数对象的严格相等，不是小于某个容差。对方阵调用 `DomainMatrix.convert_to(QQ).lu_solve`：若矩阵奇异会抛出异常；精确 LU 成功就认证满秩与解唯一。随后严格零残差证明展示的分数矩阵正是该唯一解。这样不用再进行一次代价很高的通用行化简求秩。 | Every zero above is strict equality of SymPy rational objects, not a tolerance check. For a square matrix, `DomainMatrix.convert_to(QQ).lu_solve` raises if the matrix is singular; successful exact LU therefore certifies full rank and uniqueness. The exact-zero residual then proves that the displayed fraction matrix is that unique solution. This avoids a second expensive generic row reduction merely to recompute the rank. |
| 最后把每个分数规范化为 `"numerator/denominator"` 字符串并计算 SHA-256；它必须等于随包提供的参考文件哈希。 | Finally canonicalize every fraction as a `"numerator/denominator"` string and compute SHA-256; it must equal the hash in the portable reference file. |


In [10]:
assert classic_matrix_has_full_exact_rank
assert reduced_map_has_full_exact_rank
truth_fraction_strings = [
    [str(U_classic_QQ[row, column]) for column in range(P)] for row in range(L)
]
truth_sha256 = hashlib.sha256(
    json.dumps(
        truth_fraction_strings,
        separators=(",", ":"),
    ).encode("ascii")
).hexdigest()
portable_reference = load_portable_exact_reference(PORTABLE_REFERENCE_PATH)
portable_truth_is_identical = truth_fraction_strings == portable_reference["voltage"]
portable_hash_is_identical = truth_sha256 == portable_reference["truth_sha256"]

certification_summary = {
    **input_certification,
    "classic_matrix_has_full_exact_rank": (classic_matrix_has_full_exact_rank),
    "reduced_map_has_full_exact_rank": reduced_map_has_full_exact_rank,
    "classic_residual_is_exact_zero": classic_residual_is_exact_zero,
    "robin_residual_is_exact_zero": robin_residual_is_exact_zero,
    "classic_robin_exactly_identical": (classic_robin_exactly_identical),
    "exact_voltage_gauge_zero": exact_voltage_gauge_zero,
    "portable_truth_is_identical": portable_truth_is_identical,
    "portable_hash_is_identical": portable_hash_is_identical,
    "truth_sha256": truth_sha256,
}
certification_summary

{'all_coordinates_are_fraction': True,
 'all_conductivities_are_fraction': True,
 'contact_impedance_is_fraction': True,
 'all_currents_are_rational': True,
 'all_current_columns_are_exactly_zero_sum': True,
 'first_five_exact_nodes': ((Fraction(0, 1), Fraction(0, 1)),
  (Fraction(11113, 16384), Fraction(2861, 65536)),
  (Fraction(10595, 16384), Fraction(13715, 65536)),
  (Fraction(9895, 16384), Fraction(20435, 65536)),
  (Fraction(8333, 16384), Fraction(29549, 65536))),
 'first_drive': [1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'classic_matrix_has_full_exact_rank': True,
 'reduced_map_has_full_exact_rank': True,
 'classic_residual_is_exact_zero': True,
 'robin_residual_is_exact_zero': True,
 'classic_robin_exactly_identical': True,
 'exact_voltage_gauge_zero': True,
 'portable_truth_is_identical': True,
 'portable_hash_is_identical': True,
 'truth_sha256': 'c9e62f9a3516b2ed31f20aa2d0bd471d41d4d81a7c7e371b412a03ba596ff2f7'}

### 8. 计算 `float64` 求解器相对于真值的误差 / Compute solver error relative to truth

候选 `float64` 数字不是用十进制文本近似转换，而是用 `Fraction.from_float` 取出该 IEEE-754 数值所代表的**精确二进制有理数**。随后在 100 位工作精度下评估：

$$
\varepsilon_{\mathrm{truth}}
=\frac{\|U_{\mathrm{float64}}-U_{\mathbb{Q}}\|_F}
{\|U_{\mathbb{Q}}\|_F}.
$$

缩放后向残差回答另一个问题：“候选解对输入的离散系统满足得有多好？”

$$
\eta
=\frac{\|T_r\widehat y-B^\mathsf{T}I\|_F}
{\|T_r\|_F\|\widehat y\|_F+\|B^\mathsf{T}I\|_F}.
$$

| 中文 | English |
|---|---|
| $\varepsilon_{\mathrm{truth}}$ 是前向误差，直接衡量求解器电压距离唯一精确解多远；$\eta$ 是后向残差，衡量候选结果满足方程的程度。残差小不保证前向误差最小，因为条件数会放大误差，所以两者必须同时报告。 | $\varepsilon_{\mathrm{truth}}$ is forward error, measuring distance to the unique exact voltage. $\eta$ is a backward residual, measuring equation satisfaction. A small residual does not guarantee the smallest forward error because conditioning can amplify error, so both must be reported. |


In [11]:
float_fixture, pyeidors_report = ensure_pyeidors_case(
    CASE_ID,
    SUITE_OUTPUT,
    regenerate=REGENERATE_FLOAT,
)
float_blocks = load_assembled_blocks(
    Path(float_fixture["case_dir"]) / "pyeidors_assembled_blocks.mat"
)
float_classic_state = build_classic_state(float_blocks)
float_classic_solution = solve_classic(
    float_classic_state,
    float_blocks.currents,
)
float_robin_state = build_robin_state(float_blocks)
float_robin_solution = solve_robin(
    float_robin_state,
    float_blocks.currents,
)
sample_float64 = float(float_classic_solution.electrode_voltage[0, 0])
sample_float64_as_exact_fraction = Fraction.from_float(sample_float64)
float_metrics = {
    "classic": exact_reference_metrics(
        float_classic_solution.electrode_voltage,
        portable_reference,
    ),
    "robin_transconductance": exact_reference_metrics(
        float_robin_solution.electrode_voltage,
        portable_reference,
    ),
}
{
    "sample_float64": sample_float64,
    "sample_float64_as_exact_fraction": (sample_float64_as_exact_fraction),
    "metrics": float_metrics,
}

{'sample_float64': 7.649039178557208,
 'sample_float64_as_exact_fraction': Fraction(4306026249286571, 562949953421312),
 'metrics': {'classic': {'truth_relative_l2': 1.3032086527364565e-15,
   'truth_max_abs': 9.345317203799345e-15,
   'exact_reduced_scaled_backward_residual': 4.5628231485871555e-17,
   'voltage_gauge_relative_residual': 6.823220871811818e-17,
   'reduced_condition_number_2_estimate': 47.697807812864},
  'robin_transconductance': {'truth_relative_l2': 1.4291676735987542e-15,
   'truth_max_abs': 1.0011451018574439e-14,
   'exact_reduced_scaled_backward_residual': 4.1461400398838096e-17,
   'voltage_gauge_relative_residual': 1.3722035304554455e-16,
   'reduced_condition_number_2_estimate': 47.697807812864}}}

## 检查 / Checks

| 检查 | 中文含义 | English meaning |
|---|---|---|
| `all_*_are_fraction/rational` | 几何、物性和电流输入没有先经过不可逆的小数舍入。 | Geometry, material, and current inputs were not first irreversibly rounded. |
| `classic_matrix_has_full_exact_rank` | 带规范的传统 CEM 系统在 $\mathbb{Q}$ 上可逆，解唯一。 | The gauged Classic system is invertible over $\mathbb{Q}$, so its solution is unique. |
| `reduced_map_has_full_exact_rank` | Robin 零和约化系统在 $\mathbb{Q}$ 上可逆。 | The Robin zero-sum reduced system is invertible over $\mathbb{Q}$. |
| `classic_residual_is_exact_zero` | 传统路径的分数解严格满足方程。 | The fractional Classic solution satisfies its equation exactly. |
| `robin_residual_is_exact_zero` | Robin 路径的分数解严格满足方程。 | The fractional Robin solution satisfies its equation exactly. |
| `classic_robin_exactly_identical` | 两种不同代数路径得到逐分数相同的 $U$。 | Two distinct algebraic routes produce fraction-by-fraction identical $U$. |
| `exact_voltage_gauge_zero` | 每个电流模式的电极电压和严格为零。 | Electrode voltages sum exactly to zero for every drive. |
| `portable_hash_is_identical` | 保存到 JSON 后的真值与本次重新计算结果逐字节一致。 | The JSON truth is byte-canonically identical to the recomputed truth. |


In [12]:
required_boolean_checks = {
    name: value
    for name, value in certification_summary.items()
    if isinstance(value, bool)
}
assert all(required_boolean_checks.values())
assert portable_reference["certification"] == {
    "exact_classic_residual_zero": True,
    "exact_robin_residual_zero": True,
    "exact_classic_robin_identical": True,
    "exact_voltage_gauge_zero": True,
}
required_boolean_checks

{'all_coordinates_are_fraction': True,
 'all_conductivities_are_fraction': True,
 'contact_impedance_is_fraction': True,
 'all_currents_are_rational': True,
 'all_current_columns_are_exactly_zero_sum': True,
 'classic_matrix_has_full_exact_rank': True,
 'reduced_map_has_full_exact_rank': True,
 'classic_residual_is_exact_zero': True,
 'robin_residual_is_exact_zero': True,
 'classic_robin_exactly_identical': True,
 'exact_voltage_gauge_zero': True,
 'portable_truth_is_identical': True,
 'portable_hash_is_identical': True}

## 后续步骤 / Next Steps

| 中文 | English |
|---|---|
| 1. 在 `CASE_ID="X01"` 下逐单元运行并展开任意分数。<br>2. 在 PyEIDORS/NGSolve Notebook 中对照同一网格图、指纹、$\sigma,z,I$。<br>3. 完整 38 案例的 Q0/Q2/Q4、均匀/非均匀电导率、8/16 电极和不同 $z$ 使用同一认证逻辑；大于等于 500 节点的 Q4 案例改用 `python-flint fmpq_mat.solve`，仍然在 $\mathbb{Q}$ 上严格求解和验证零残差。<br>4. 若研究连续 PDE 误差，应另做真实圆域加密实验，不能把本 Notebook 的离散真值误称为连续解析解。 | 1. Run cell-by-cell with `CASE_ID="X01"` and expand any fraction.<br>2. Cross-check the same mesh plot, fingerprint, $\sigma,z,I$ in the PyEIDORS and NGSolve notebooks.<br>3. All 38 Q0/Q2/Q4, uniform/heterogeneous conductivity, 8/16-electrode, and contact-impedance cases use the same certification logic. Q4 cases with at least 500 nodes use `python-flint fmpq_mat.solve`, still solving over $\mathbb{Q}$ with exact-zero residual checks.<br>4. Study continuum-PDE error separately on a true-disk refinement sequence; do not call this notebook's discrete truth a continuum analytic solution. |
